# Phase 0: Dataset Audit
This notebook performs the initial dataset audit as specified in `AGENTS.md`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid")

# Path to the data
DATA_PATH = "../data/raw/connections_princeton.csv.gz"

print(f"Loading data from {DATA_PATH}...")
df = pd.read_csv(DATA_PATH)
print("Data loaded successfully!")


## 1. Basic Information
- row count
- columns
- dtypes
- missing values
- duplicate rows


In [ ]:
print("--- Basic Information ---")
print(f"Row count (raw download rows): {len(df)}")
print("\nColumns and Data Types:")
print(df.dtypes)

print("\nMissing values per column:")
print(df.isnull().sum())

print("\nChecking for duplicate rows (this may take a moment)...")
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")


## 2. Graph Structure
- unique presynaptic neurons
- unique postsynaptic neurons
- unique neurons overall
- unique `(pre, post)` pairs
- connection counts at several synapse thresholds


In [ ]:
print("--- Graph Structure ---")
unique_pre = df['pre_root_id'].nunique()
unique_post = df['post_root_id'].nunique()
unique_all = pd.concat([df['pre_root_id'], df['post_root_id']]).nunique()

print(f"Unique presynaptic neurons: {unique_pre}")
print(f"Unique postsynaptic neurons: {unique_post}")
print(f"Unique neurons overall: {unique_all}")

# Group by pre and post to find unique neuron pairs and calculate aggregated syn_count
print("\nCalculating unique (pre, post) pairs...")
grouped_pairs = df.groupby(['pre_root_id', 'post_root_id'])['syn_count'].sum().reset_index()

unique_pairs = len(grouped_pairs)
print(f"Unique (pre, post) pairs (thresholded graph edges > 0): {unique_pairs}")

print("\nConnection counts at several synapse thresholds:")
thresholds = [1, 2, 5, 10, 50, 100]
for t in thresholds:
    count = (grouped_pairs['syn_count'] >= t).sum()
    print(f"  Edges with syn_count >= {t}: {count}")


## 3. Weights (syn_count)
- sum
- mean
- median
- standard deviation
- min/max
- percentiles
- histogram


In [ ]:
print("--- Weights (syn_count) ---")

print("Stats based on unique neuron pairs (aggregated edges):")
syn_desc = grouped_pairs['syn_count'].describe(percentiles=[.25, .5, .75, .90, .95, .99])
print(syn_desc)
print(f"Sum of all synapses: {grouped_pairs['syn_count'].sum()}")

# Histogram
plt.figure(figsize=(10, 5))
# Since the distribution is highly skewed, we use a log scale
sns.histplot(grouped_pairs['syn_count'], bins=50, log_scale=(False, True))
plt.title("Distribution of Synapse Counts per Unique Edge (Log Scale for Frequency)")
plt.xlabel("Synapse Count")
plt.ylabel("Frequency (Log Scale)")
plt.show()


## 4. Degree
- in-degree
- out-degree
- weighted in-degree
- weighted out-degree


In [ ]:
print("--- Degree ---")
# out-degree: number of unique post partners for each pre
out_degree = grouped_pairs.groupby('pre_root_id').size()
# in-degree: number of unique pre partners for each post
in_degree = grouped_pairs.groupby('post_root_id').size()

print("Out-degree (number of unique outgoing partners):")
print(out_degree.describe())

print("\nIn-degree (number of unique incoming partners):")
print(in_degree.describe())

# Weighted degree
weighted_out_degree = grouped_pairs.groupby('pre_root_id')['syn_count'].sum()
weighted_in_degree = grouped_pairs.groupby('post_root_id')['syn_count'].sum()

print("\nWeighted Out-degree (total outgoing synapses):")
print(weighted_out_degree.describe())

print("\nWeighted In-degree (total incoming synapses):")
print(weighted_in_degree.describe())


## 5. Directionality
- reciprocal pair count
- reciprocal fraction
- self-loop count


In [ ]:
print("--- Directionality ---")

# Self loops
self_loops = grouped_pairs[grouped_pairs['pre_root_id'] == grouped_pairs['post_root_id']]
print(f"Self-loop count (unique edges where pre == post): {len(self_loops)}")

# Reciprocity
print("\nCalculating reciprocal pairs...")
reciprocal_edges = pd.merge(
    grouped_pairs, 
    grouped_pairs, 
    left_on=['pre_root_id', 'post_root_id'], 
    right_on=['post_root_id', 'pre_root_id']
)
reciprocal_pair_count = len(reciprocal_edges) // 2
reciprocal_fraction = len(reciprocal_edges) / unique_pairs

print(f"Reciprocal pair count (undirected): {reciprocal_pair_count}")
print(f"Fraction of unique directed edges that participate in a reciprocal relationship: {reciprocal_fraction:.4f}")


## 6. Metadata
- neuropil counts
- neurotransmitter counts


In [ ]:
print("--- Metadata ---")

print("Neuropil distribution (top 15):")
print(df['neuropil'].value_counts().head(15))

print("\nNeurotransmitter distribution:")
print(df['nt_type'].value_counts(dropna=False))

# Visualization
plt.figure(figsize=(10, 5))
df['nt_type'].value_counts().plot(kind='bar')
plt.title("Neurotransmitter Predictions (Raw row counts)")
plt.xlabel("nt_type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()
